# 第14章 利率期货与远期 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch14_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch14_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 7：例14.1/14.2 + CTD 是否切换


In [ ]:
from fi import futures as fut
for c,yrs in [(0.028,7),(0.032,8),(0.030,9)]: print(f'CF(票息{c*100:.1f}%,{yrs}y)={fut.conversion_factor(c,yrs):.4f}')
def ctd_at(F):
    bonds = [{'name':'A','clean_price':99.10,'conversion_factor':fut.conversion_factor(0.028,7)},
             {'name':'B','clean_price':101.55,'conversion_factor':fut.conversion_factor(0.032,8)},
             {'name':'C','clean_price':100.30,'conversion_factor':fut.conversion_factor(0.030,9)}]
    best,_ = fut.ctd(bonds, F); return best['name']
for F in (98,100,103): print(f'期货价={F}: CTD={ctd_at(F)}')
print('期货价变化可能触发 CTD 切换')


## 编程实验 8：久期中性套保 + 图14-1


In [ ]:
import numpy as np
from fi import plotting
plotting.use_chinese_style()
port_dv01 = 5*1e8*1e-4
ctd_dv01 = fut.bond_dv01(0.032,8,0.032); ctd_cf = fut.conversion_factor(0.032,8)
contract_dv01 = fut.futures_dv01(ctd_dv01,ctd_cf)/100*1e6
N = round(-port_dv01/contract_dv01)
print(f'组合DV01={port_dv01:.0f} 每张期货DV01={contract_dv01:.1f} -> 卖出{abs(N)}张')
net = port_dv01 + N*contract_dv01; sh = np.linspace(-100,100,41)
fig, ax = plotting.new_axes()
ax.plot(sh, -port_dv01*sh/1e4, label='未对冲'); ax.plot(sh, -net*sh/1e4, label=f'对冲(卖{abs(N)}张)')
ax.axhline(0, ls=':', color='gray'); ax.set_xlabel('利率冲击 (bp)'); ax.set_ylabel('损益(万元)'); ax.set_title('久期中性套保'); ax.legend(); fig.tight_layout()


## 编程实验 9：FRA + QuantLib 远期利率


In [ ]:
print('FRA 多头价值 =', round(fut.fra_value(1e8,0.025,0.028,0.25),0), '元')
import QuantLib as ql
today = ql.Date(15,6,2026); ql.Settings.instance().evaluationDate = today; dc = ql.Actual365Fixed()
yc = ql.YieldTermStructureHandle(ql.FlatForward(today,0.025,dc))
d1,d2 = today+ql.Period(1,ql.Years), today+ql.Period(15,ql.Months)
print('QuantLib 1y3m 远期 =', round(yc.forwardRate(d1,d2,dc,ql.Simple).rate()*100,4), '% (FRA公允约定利率)')
